# 05 – Kueri Sintetis (Known-Item)

Masukan: `songs_clean.parquet` dan `lyrecall_text.py` (notebook 01).
Keluaran: `queries.parquet`, `queries_summary.json`.

Rancangan (Tabel 2 laporan):
- Fragmen = 2–4 baris berurutan dari satu lagu target, posisi acak (tidak harus sejajar dengan batas chunk).
- **exact**: fragmen apa adanya (huruf kecil, tanpa tanda baca, sama seperti pengguna mengetik).
- **parsial**: ±30% kata dibuang secara acak (minimal 3 kata tersisa).
- **berisik**: substitusi kata dengan kata yang mirip bunyinya (kamus pengucapan CMU: homofon atau beda satu fonem)
  lalu salah ketik (ganti dengan tombol tetangga QWERTY, hapus, tukar, atau sisip huruf).
  Kolom `q_sound` (hanya salah dengar) dan `q_typo` (hanya salah ketik) disimpan untuk analisis.
- Ketiga jenis dibuat dari **fragmen yang sama** → perbandingan berpasangan antar retriever dan antar jenis kueri.
- Kueri **ambigu** (fragmen yang sama persis ada di lebih dari satu lagu, mis. refren atau cover) dibuang.
- Lagu **validasi** (untuk menyetel k1, b, k RRF, dll.) dan lagu **uji** (hasil akhir) dipisah sejak awal dan tidak beririsan.
- Kueri parafrase (opsional) tidak dibuat di sini.

In [1]:
# Konfigurasi dan data
import os, sys, re, glob, json, time
from collections import Counter, defaultdict
import numpy as np, pandas as pd

try:
    import cmudict
except ImportError:
    !pip -q install cmudict
    import cmudict

sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/lyrecall_text.py", recursive=True)[0]))
from lyrecall_text import normalize_lex

# ====== KONFIGURASI ======
SEED          = 42
N_VAL         = 1_000          # kueri validasi (penyetelan parameter)
N_TEST        = 3_000          # kueri uji (hasil akhir)
LINE_CHOICES  = (1, 2, 3, 4)      # jumlah baris fragmen
MIN_Q_WORDS, MAX_Q_WORDS, MIN_UNIQUE = 6, 50, 4
PARTIAL_DROP  = 0.30           # porsi kata yang dibuang pada kueri parsial
P_SOUND       = 0.10           # peluang kata diganti kata mirip bunyi
P_TYPO        = 0.10           # peluang kata diberi salah ketik
MIN_CNT_SUBST = 100            # kata pengganti harus muncul >= 100x di korpus
MAX_TRIES     = 8              # percobaan fragmen per lagu
SPARE         = 1.3            # lagu cadangan jika ada yang gagal
OUT_DIR       = "/kaggle/working"

path = glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/songs_clean.parquet", recursive=True)[0]
songs = pd.read_parquet(path, columns=["song_id", "lyrics"])
print("Lagu:", len(songs))

Lagu: 99821


In [2]:
# Baris ternormalisasi dan frekuensi data
t0 = time.time()
nlines, wc = [], Counter()
for lyr_ in songs["lyrics"]:
    ls = [normalize_lex(l) for l in lyr_.split("\n")]
    ls = [l for l in ls if l]
    nlines.append(ls)
    wc.update(" ".join(ls).split())
song_ids = songs["song_id"].to_numpy()
print(f"{len(nlines):,} lagu | kosakata {len(wc):,} | {time.time()-t0:.0f} dtk")

99,821 lagu | kosakata 155,138 | 54 dtk


## Model kesalahan: mirip bunyi dan salah ketik
- Mirip bunyi: dua kata bertetangga bila pengucapan CMU-nya identik (homofon) atau berbeda satu fonem
  (indeks "hapus satu fonem"). Pengganti dipilih acak dengan bobot akar frekuensi kata.
- Salah ketik: satu suntingan per kata terpilih (ganti dengan tombol tetangga, hapus, tukar, atau sisip).

In [3]:
# Indeks fonetik dan fungsi derau
cm = cmudict.dict()
PH = {}
for w, prons in cm.items():
    key = w.replace("'", "")
    if wc.get(key, 0) < MIN_CNT_SUBST or not key.isascii() or not key.isalpha():
        continue
    if key in PH and "'" in w:          # utamakan bentuk tanpa apostrof ("ill" bukan "i'll")
        continue
    PH[key] = tuple(re.sub(r"\d", "", x) for x in prons[0])

BY_PH = defaultdict(list)     # pengucapan -> kata (homofon)
SUBS  = defaultdict(set)      # (posisi, pengucapan tanpa fonem di posisi itu) -> kata (ganti 1 fonem)
INS   = defaultdict(set)      # pengucapan yang lebih pendek -> kata yang lebih panjang (tambah 1 fonem)
for w, ph in PH.items():
    if len(ph) < 3:
        continue
    BY_PH[ph].append(w)
    for i in range(len(ph)):
        v = ph[:i] + ph[i + 1:]
        SUBS[(i, v)].add(w)
        INS[v].add(w)

_cache = {}
def sound_alikes(w):
    if w in _cache:
        return _cache[w]
    ph = PH.get(w)
    if ph is None or len(ph) < 3:
        _cache[w] = ([], [])
        return _cache[w]
    homo = [x for x in BY_PH[ph] if x != w]
    near = set()
    for i in range(len(ph)):
        v = ph[:i] + ph[i + 1:]
        near |= SUBS.get((i, v), set())      # ganti satu fonem
        near |= set(BY_PH.get(v, []))        # kata lebih pendek (hapus satu fonem)
    near |= INS.get(ph, set())               # kata lebih panjang (tambah satu fonem)
    near.discard(w); near -= set(homo)
    _cache[w] = (homo, sorted(near))
    return _cache[w]

def pick_sound(w, rng):
    homo, near = sound_alikes(w)
    pool = homo if (homo and rng.random() < 0.5) else (near or homo)
    if not pool:
        return None
    wts = np.array([wc[x] for x in pool], dtype=float) ** 0.5
    return pool[int(rng.choice(len(pool), p=wts / wts.sum()))]

# --- salah ketik: tetangga tombol QWERTY ---
rows_kb = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
offs = [0.0, 0.25, 0.75]
POS = {ch: (r, c + offs[r]) for r, row in enumerate(rows_kb) for c, ch in enumerate(row)}
KB = {a: [b for b in POS if b != a and
          (POS[a][0] - POS[b][0]) ** 2 + (POS[a][1] - POS[b][1]) ** 2 <= 2.0] for a in POS}

def typo(word, rng):
    chars = list(word)
    i = int(rng.integers(len(chars)))
    op = str(rng.choice(["sub", "del", "swap", "ins"], p=[0.4, 0.25, 0.2, 0.15]))
    if op == "del" and len(chars) > 3:
        del chars[i]
    elif op == "swap" and len(chars) > 1:
        j = min(i, len(chars) - 2)
        chars[j], chars[j + 1] = chars[j + 1], chars[j]
    elif op == "ins":
        chars.insert(i, str(rng.choice(KB.get(chars[i], list("etaoin")))))
    elif chars[i] in KB:
        chars[i] = str(rng.choice(KB[chars[i]]))
    return "".join(chars)

def _try_typo(w, rng):
    for _ in range(5):
        t = typo(w, rng)
        if t and t != w:
            return t
    return None

def apply_typos(words, rng, p=P_TYPO):
    out, n = list(words), 0
    elig = [i for i, w in enumerate(words) if len(w) >= 4 and w.isalpha()]
    for i in elig:
        if rng.random() < p:
            t = _try_typo(words[i], rng)
            if t:
                out[i] = t; n += 1
    if n == 0 and elig:                     # pastikan minimal satu perubahan
        i = int(rng.choice(elig))
        t = _try_typo(words[i], rng)
        if t:
            out[i] = t; n = 1
    return out, n

def apply_sound(words, rng, p=P_SOUND):
    out, n = list(words), 0
    elig = [i for i, w in enumerate(words) if any(sound_alikes(w))]
    for i in elig:
        if rng.random() < p:
            s = pick_sound(words[i], rng)
            if s:
                out[i] = s; n += 1
    if n == 0 and elig:
        i = int(rng.choice(elig))
        s = pick_sound(words[i], rng)
        if s:
            out[i] = s; n = 1
    return out, n

def make_partial(words, rng):
    n = len(words)
    k = min(int(round(PARTIAL_DROP * n)), n - 3)
    drop = set(rng.choice(n, size=k, replace=False).tolist())
    return [w for i, w in enumerate(words) if i not in drop]

# --- cek ---
print("Kata dengan pengucapan:", f"{len(PH):,}")
for w in ["night", "love", "dont", "baby"]:
    h, nr = sound_alikes(w)
    print(f"  {w}: homofon {h[:5]} | mirip {nr[:8]}")
print("Tetangga tombol 'g':", KB["g"])

Kata dengan pengucapan: 8,427
  night: homofon ['knight', 'nite'] | mirip ['bite', 'fight', 'height', 'kite', 'knife', 'knights', 'knot', 'light']
  love: homofon [] | mirip ['dove', 'glove', 'leave', 'live', 'loved', 'lover', 'loves', 'luck']
  dont: homofon [] | mirip ['wont']
  baby: homofon [] | mirip ['babe', 'babes', 'babies', 'babys', 'bobby', 'maybe']
Tetangga tombol 'g': ['t', 'y', 'f', 'h', 'v', 'b']


## Kandidat fragmen dan pembuangan kueri ambigu
Tiap lagu terpilih diberi hingga `MAX_TRIES` fragmen kandidat. Hash tiap jendela baris di seluruh korpus
dihitung untuk mencari fragmen yang sama persis di lagu lain.

In [4]:
# Kandidat dan hitung ambiguitas
t0 = time.time()
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(nlines))
pool_songs = perm[: int((N_VAL + N_TEST) * SPARE)]

all_cands = {}
for si in pool_songs:
    ls, cands = nlines[si], []
    for _ in range(MAX_TRIES):
        k = int(rng.choice(LINE_CHOICES))
        if len(ls) < k:
            continue
        s = int(rng.integers(0, len(ls) - k + 1))
        fl = ls[s:s + k]
        words = " ".join(fl).split()
        if not (MIN_Q_WORDS <= len(words) <= MAX_Q_WORDS) or len(set(words)) < MIN_UNIQUE:
            continue
        cands.append((k, s, words, hash(" ".join(fl))))
    all_cands[si] = cands
print("Lagu kandidat:", len(all_cands), "| tanpa kandidat valid:",
      sum(1 for c in all_cands.values() if not c))

need = {k: set() for k in LINE_CHOICES}
for cands in all_cands.values():
    for (k, s, words, h) in cands:
        need[k].add(h)

cnt = {k: Counter() for k in LINE_CHOICES}
for ls in nlines:                                   # satu kali lewat seluruh korpus
    for k in LINE_CHOICES:
        if len(ls) < k:
            continue
        seen = set()
        for s in range(len(ls) - k + 1):
            h = hash(" ".join(ls[s:s + k]))
            if h in need[k]:
                seen.add(h)
        for h in seen:
            cnt[k][h] += 1

n_cand = sum(len(c) for c in all_cands.values())
n_amb = sum(1 for c in all_cands.values() for (k, s, w, h) in c if cnt[k][h] > 1)
print(f"Kandidat: {n_cand:,} | ambigu (muncul di > 1 lagu): {n_amb:,} ({n_amb / max(n_cand, 1):.1%}) "
      f"| {time.time()-t0:.0f} dtk")

Lagu kandidat: 5200 | tanpa kandidat valid: 9
Kandidat: 36,548 | ambigu (muncul di > 1 lagu): 306 (0.8%) | 11 dtk


In [5]:
# Rakit kueri
rows, n_val, n_test = [], 0, 0
for si in pool_songs:
    if n_val >= N_VAL and n_test >= N_TEST:
        break
    good = [c for c in all_cands[si] if cnt[c[0]][c[3]] == 1]     # tidak ambigu
    if not good:
        continue
    split = "val" if n_val < N_VAL else "test"
    k, s, words, _ = good[0]
    w_sound, n_s = apply_sound(words, rng)
    w_typo, n_t = apply_typos(words, rng)
    w_noisy, n_tn = apply_typos(w_sound, rng)
    if " ".join(w_noisy) == " ".join(words):
        continue
    rows.append({
        "qid": f"{split}-{(n_val if split == 'val' else n_test):05d}", "split": split,
        "song_id": int(song_ids[si]), "k_lines": k, "start_line": s, "n_words": len(words),
        "q_exact": " ".join(words), "q_partial": " ".join(make_partial(words, rng)),
        "q_noisy": " ".join(w_noisy), "q_typo": " ".join(w_typo), "q_sound": " ".join(w_sound),
        "n_sound": n_s, "n_typo": n_t, "n_edit_noisy": n_s + n_tn,
    })
    if split == "val":
        n_val += 1
    else:
        n_test += 1

Q = pd.DataFrame(rows)
print(Q["split"].value_counts().to_dict())

{'test': 3000, 'val': 1000}


## Pemeriksaan

In [6]:
# Statistik dan contoh
assert set(Q[Q.split == "val"].song_id).isdisjoint(set(Q[Q.split == "test"].song_id)), "val/test beririsan"
assert Q["qid"].is_unique

print("Kueri per split      :", Q["split"].value_counts().to_dict())
print("Jumlah baris fragmen :", Q["k_lines"].value_counts().sort_index().to_dict())
print("Kata per kueri exact :", Q["n_words"].describe().round(1).to_dict())
print("Noisy sama dgn exact :", int((Q.q_noisy == Q.q_exact).sum()))
print("Tanpa substitusi bunyi (n_sound=0):", int((Q.n_sound == 0).sum()))
print("Rerata suntingan pada noisy       :", round(Q.n_edit_noisy.mean(), 2))
print("Rerata kata parsial / exact       :",
      round((Q.q_partial.str.split().str.len() / Q.n_words).mean(), 2))

for _, r in Q.sample(5, random_state=1).iterrows():
    print(f"\n[{r.qid}] {r.k_lines} baris | lagu {r.song_id}")
    print("  exact  :", r.q_exact)
    print("  parsial:", r.q_partial)
    print("  berisik:", r.q_noisy)

Kueri per split      : {'test': 3000, 'val': 1000}
Jumlah baris fragmen : {1: 655, 2: 1054, 3: 1192, 4: 1099}
Kata per kueri exact : {'count': 4000.0, 'mean': 18.5, 'std': 9.1, 'min': 6.0, '25%': 11.0, '50%': 17.0, '75%': 24.0, 'max': 50.0}
Noisy sama dgn exact : 0
Tanpa substitusi bunyi (n_sound=0): 7
Rerata suntingan pada noisy       : 2.79
Rerata kata parsial / exact       : 0.7

[val-00200] 3 baris | lagu 87542
  exact  : mia fey there are times when youll feel lonely and defeated and throughout some trials youll only be mistreated
  parsial: mia fey times when lonely and and throughout some trials only be mistreated
  berisik: mia fey there are times when mul feel lonely and defeated and throughout sum trials youll oly be mistreated

[test-00078] 4 baris | lagu 5419
  exact  : 10 of us show guess who won we hung out on 7th and a friends worked the door we didnt have to pay
  parsial: 10 us show guess who hung on 7th a friends worked the door we didnt have
  berisik: 10 of us show 

In [7]:
# Simpan
Q.to_parquet(f"{OUT_DIR}/queries.parquet", index=False)
summary = {
    "seed": SEED, "n_val": int((Q.split == "val").sum()), "n_test": int((Q.split == "test").sum()),
    "line_choices": list(LINE_CHOICES), "min_q_words": MIN_Q_WORDS, "max_q_words": MAX_Q_WORDS,
    "partial_drop": PARTIAL_DROP, "p_sound": P_SOUND, "p_typo": P_TYPO,
    "candidates": n_cand, "ambiguous_candidates": n_amb,
    "ambiguous_rate": round(n_amb / max(n_cand, 1), 4),
    "noisy_equal_exact": int((Q.q_noisy == Q.q_exact).sum()),
    "no_sound_substitution": int((Q.n_sound == 0).sum()),
    "mean_edits_noisy": round(float(Q.n_edit_noisy.mean()), 2),
    "n_songs": len(nlines),
}
with open(f"{OUT_DIR}/queries_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "seed": 42,
  "n_val": 1000,
  "n_test": 3000,
  "line_choices": [
    1,
    2,
    3,
    4
  ],
  "min_q_words": 6,
  "max_q_words": 50,
  "partial_drop": 0.3,
  "p_sound": 0.1,
  "p_typo": 0.1,
  "candidates": 36548,
  "ambiguous_candidates": 306,
  "ambiguous_rate": 0.0084,
  "noisy_equal_exact": 0,
  "no_sound_substitution": 7,
  "mean_edits_noisy": 2.79,
  "n_songs": 99821
}
